# Programmes to Data
In this notebook:

1. Ingest theatre programmes as PDF and convert to images.
2. Use a VLM to transcribe them with better performance than traditional OCR.
3. Use an LLM to convert this full text to structured data using the [Linked Art](https://linked.art/) ontology with the [LA-PA extension](https://github.com/stage-to-data/linked-art-pa) for performing arts.

_Part of the ERC-funded STAGE project. (Find out more [here](https://stage-to-data.huma-num.fr/))._

## 1. Setup
Mandatory grotesque setup cell to install necessary libraries and download necessary data.

__Running this cell will take a few minutes and spit a lot of junk to the console.__

In [ ]:
!pip install -q --disable-pip-version-check git+https://github.com/stage-to-data/llm-wrap.git
!pip install -q --disable-pip-version-check git+https://github.com/stage-to-data/programmes-to-data-example.git
import ptod
import os
import llmwrap
from google.colab import userdata

print("👍 Ready!")

## 2. Prepare sources
First, we need to get our sources ready for processing. We will drop all of the PDF files we wish to process in a folder, and give th epath to this folder as the `TO_PROCESS` variable. At the same time we'll set an `OUTPUT_FOLDER` to store all of the results of our processes.

In [ ]:
TO_PROCESS = os.path.join(os.getcwd(), "sources")
OUTPUT_FOLDER = os.path.join(os.getcwd(), "outputs")
if os.path.isdir(OUTPUT_FOLDER) == False:
    os.makedirs(OUTPUT_FOLDER, exist_ok = True)

pdf_files = ptod.collect_files(TO_PROCESS, ["pdf"])
print(f"👍 Found {len(pdf_files)} PDF files to process.")

Next, we need to convert these PDF files to images (so that they can be accepted by the VLM) and also perform some light preprocessing so that they will produce better results (converting to greyscale and reducing size).

In [ ]:
print("Converting to image...")
for pdf_file in pdf_files:
    ptod.pdf_to_img(pdf_file, os.path.join(OUTPUT_FOLDER, "images", os.path.splitext(os.path.basename(pdf_file))[0], "raw-png"))

print("Preprocessing...")
for pdf_file in pdf_files:
    ptod.preprocess_images(
        pdf_file, OUTPUT_FOLDER,
        color_mode = "L",
        boost_contrast = False,
        contrast_amount = 2.0,
        max_dims = 3000,
        max_image_size = 5)
    
print("👍 Done!")

## 3. VLM Transcription
Now we can take these images, and use a computer vision model to get a transcription of their textual content.

In [ ]:
api_key = userdata.get("API_KEY")

for pdf_file in pdf_files:
    ptod.transcribe(
        pdf_file, 
        OUTPUT_FOLDER, 
        api_key = api_key
    )

print("👍 Done!")

## 4. Transcription stucturation
Now we can take these transcriptions and run them through an LLM that has been specifically trained to extract structered data in the Linked Art ontology. This can be done using a different models: either Method 1 using the Pleias model, or Method 2 using Claude.

In [ ]:
# Method 1 (Pleias)

pleias_model = ptod.PleiasModel(
    model_id = "Pclanglais/POntAvignon-4b"
)

for pdf_file in pdf_files:
    ptod.extract_data(pleias_model, pdf_file, OUTPUT_FOLDER)

print("👍 Done!")

In [ ]:
# Method 2 (Claude)

for pdf_file in pdf_files:
    ptod.extract_data_claude(pdf_file, OUTPUT_FOLDER, api_key = api_key)

print("👍 Done!")